# Exploring a local copy of Tahoe-100M

[Tahoe-100M](https://huggingface.co/datasets/tahoebio/Tahoe-100M) holds about 96 million
single-cell transcriptomes of 50 cancer cell lines. All lines are pooled in every well, and the
wells were treated with 379 drugs at three doses on 14 plates (96 wells each, two of them DMSO
controls). This notebook works with a full local download without loading it into memory:

1. **Metadata**: genes, wells (plate, drug, dose), drugs and cell lines.
2. **Overview** of all cells: cells per plate, cell line and well, and a cell-count view of drug
   sensitivity.
3. **Per-cell QC** from `obs_metadata` (optional).
4. **Select cells** by organ, cell line, drug, dose and plate, sample a balanced subset and load
   its counts as an AnnData.
5. **Visualise** the subset: QC, UMAP, and drug effects relative to the same plate's DMSO wells.
6. **Export** the subset as `.h5ad`, and optionally as a tokenized TERRA dataset for alignment
   fine-tuning (see `docs/tahoe_alignment.md`).

The helpers come from `terra.tahoe.local` and `terra.tahoe.convert`.

## Getting the data

Set `TAHOE_DIR` below to the folder holding either layout.

**Parquet shards (recommended).** About 340 GB, no conversion, and well filters skip most of the
data using the shards' row-group statistics:

```python
from huggingface_hub import snapshot_download

snapshot_download(
    "tahoebio/Tahoe-100M", repo_type="dataset", local_dir="/data/tahoe",
    allow_patterns=["data/*.parquet"] + [f"metadata/{name}.parquet" for name in
                    ["gene_metadata", "sample_metadata", "drug_metadata",
                     "cell_line_metadata", "obs_metadata"]])
```

**`datasets` cache.** `datasets.load_dataset("tahoebio/Tahoe-100M", split="train",
cache_dir="/data/tahoe")` also works (the keyword is `streaming=`, not `stream=`). It converts
the parquet into uncompressed Arrow files in `cache_dir`: about 1.8 TB (5.3 times the parquet
size, measured on one shard). With `datasets` 4.x, the downloaded parquet (338 GB) stays in the
Hugging Face hub cache, `~/.cache/huggingface/hub` unless `HF_HUB_CACHE` is set. Its
`datasets--tahoebio--Tahoe-100M` folder is itself a valid `TAHOE_DIR`.

The small metadata tables are looked up under `TAHOE_DIR` and downloaded into it when missing.
`obs_metadata` (2.3 GB) is only needed for section 3.

**Memory.** Sections 1-3 need a few GB. Section 4 holds the metadata of the matching cells and
the counts of the sampled ones. The defaults (lung lines, five drugs, one plate) load about
20,000 cells.

In [ ]:
import os
from pathlib import Path

# Folder with the download: parquet shards (snapshot_download, or the hub cache's
# datasets--tahoebio--Tahoe-100M folder) or a `datasets` cache_dir.
TAHOE_DIR = Path(os.environ.get("TAHOE_DIR", "/path/to/tahoe"))
# Outputs: cached cell counts and the exported subset.
OUT_DIR = Path(os.environ.get("TAHOE_OUT", "tahoe_explore"))

# Section 3: per-cell QC from obs_metadata (2.3 GB, downloaded if absent).
USE_OBS_METADATA = True
OBS_ROW_GROUP_STRIDE = 4  # read every 4th of its 96 row groups (~25M cells)

# Section 4: cells to load (None = no restriction).
ORGANS = ["Lung"]
CELL_NAMES = None  # e.g. ["A549", "NCI-H358"]
DRUGS = ["Trametinib", "Paclitaxel", "Bortezomib", "Gemcitabine", "Afatinib"]
DOSES_UM = [5.0]  # 0.05, 0.5 or 5.0
PLATES = None
INCLUDE_CONTROLS = True  # add the DMSO wells of the selected wells' plates
GROUP_BY = ["cell_line_id", "sample"]  # balance cells per cell line and well
CELLS_PER_GROUP = 200
SEED = 0

# Section 6: tokenized TERRA dataset for alignment fine-tuning (None: skip).
TERRA_MODEL_DIR = os.environ.get("TERRA_MODEL_DIR")  # e.g. the TERRA-96M snapshot folder
LINEAGE_MAP = Path(os.environ.get("LINEAGE_MAP", "../../configs/finetune/lineage_map.yaml"))
TERRA_LAYOUT = {"seq_len_cell": 2048, "seq_len_neigh": 256, "model_input_size": 4608}

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import logging
import sys
import time

# torch (imported by terra) and numba (scanpy's neighbours and UMAP) each load an
# OpenMP runtime, and on macOS the two crash together (segmentation fault in
# sc.pp.neighbors). numba's own thread pool avoids that; set before scanpy loads.
if sys.platform == "darwin":
    os.environ.setdefault("NUMBA_THREADING_LAYER", "workqueue")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import scanpy as sc
import seaborn as sns

from terra.tahoe.convert import CONTROL_DRUG, cell_line_table, gene_lookup, sample_table, table_to_anndata
from terra.tahoe.local import (
    cell_counts,
    cells_filter,
    metadata_file,
    open_expression_data,
    read_cells,
    read_obs,
    sample_per_group,
    select_cell_lines,
    select_samples,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sc.settings.set_figure_params(dpi=80, frameon=False)


def plate_order(plates):
    """Plate names in numeric order: plate1, plate2, ..., plate14."""
    return sorted(set(plates), key=lambda plate: int(plate.removeprefix("plate")))

## 1. Metadata

`sample_metadata` has one row per well (`sample`), with its plate, drug and dose.
`cell_line_metadata` has one row per driver mutation, so `cell_line_table` reduces it to one row
per line, keyed by the Cellosaurus id that the expression data calls `cell_line_id`.

In [ ]:
def read_metadata(name):
    """A Tahoe-100M metadata table, from TAHOE_DIR or downloaded into it."""
    return pd.read_parquet(metadata_file(name, TAHOE_DIR))


gene_meta = read_metadata("gene_metadata")
sample_meta = read_metadata("sample_metadata")
drug_meta = read_metadata("drug_metadata")
cell_line_meta = read_metadata("cell_line_metadata")

wells = sample_table(sample_meta).join(sample_meta.set_index("sample")[["plate", "drug"]])
wells["drug"] = wells["drug"].str.strip()
lines = cell_line_table(cell_line_meta)
print(f"{len(gene_meta):,} genes; {len(wells):,} wells on {wells['plate'].nunique()} plates; "
      f"{wells['drug'].nunique() - 1} drugs + DMSO; {len(lines)} cell lines in the metadata")
wells.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
(wells.assign(dose=wells["dose_uM"].map(lambda dose: f"{dose:g} µM"))
      .groupby(["plate", "dose"]).size().unstack(fill_value=0)
      .loc[plate_order(wells["plate"])]
      .plot.bar(stacked=True, ax=axes[0], xlabel="", title="Wells per plate and dose"))
drug_meta["moa-broad"].value_counts().plot.barh(ax=axes[1], title="Drugs by broad mechanism")
lines["organ"].value_counts().sort_values().plot.barh(ax=axes[2], title="Cell lines in the metadata, by organ")
plt.tight_layout()
plt.show()

## 2. Overview of all cells

Opening the data reads no cells: parquet shards are read lazily and Arrow files are
memory-mapped. `cell_counts` reads only the metadata columns, once, and counts the cells per
plate, well and cell line. For all cells this takes minutes, so the result is cached in
`OUT_DIR`.

In [ ]:
dataset = open_expression_data(TAHOE_DIR)
print(f"{dataset.count_rows():,} cells")
print(dataset.schema.to_string(show_schema_metadata=False))

In [ ]:
counts_file = OUT_DIR / "cell_counts.parquet"  # delete it to recount (e.g. after changing TAHOE_DIR)
if counts_file.exists():
    counts = pd.read_parquet(counts_file)
else:
    start = time.time()
    counts = cell_counts(dataset, by=["plate", "sample", "cell_line_id"])
    counts.to_parquet(counts_file)
    print(f"Counted in {time.time() - start:.0f} s")

counts = (counts.join(wells[["drug", "dose_uM"]], on="sample")
                .join(lines[["cell_name", "organ"]], on="cell_line_id"))
counts["is_control"] = counts["drug"].eq(CONTROL_DRUG)
print(f"{counts['n_cells'].sum():,} cells: {counts['cell_line_id'].nunique()} cell lines, "
      f"{counts['sample'].nunique()} wells, {counts['plate'].nunique()} plates")
counts.head()

In [ ]:
plates = plate_order(counts["plate"])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
(counts.groupby("plate")["n_cells"].sum().loc[plates].div(1e6)
       .plot.bar(ax=axes[0], xlabel="", ylabel="million cells", title="Cells per plate"))
per_well = counts.groupby(["sample", "is_control"], as_index=False)["n_cells"].sum()
for is_control, part in per_well.groupby("is_control"):
    axes[1].hist(part["n_cells"], bins=40, alpha=0.7, label="DMSO" if is_control else "treated")
axes[1].set(title="Cells per well", xlabel="cells", ylabel="wells")
axes[1].legend()
plt.tight_layout()
plt.show()

grid = (counts.pivot_table(index="cell_name", columns="plate", values="n_cells",
                           aggfunc="sum", fill_value=0)
              .reindex(columns=plates, fill_value=0))
grid = grid.loc[grid.sum(axis=1).sort_values().index]
organ_of = lines.drop_duplicates("cell_name").set_index("cell_name")["organ"]
fig, ax = plt.subplots(figsize=(10, max(4, 0.22 * len(grid))))
image = ax.imshow(np.log10(grid + 1), aspect="auto", cmap="viridis")
ax.set_xticks(range(len(plates)), plates, rotation=90)
ax.set_yticks(range(len(grid)), [f"{name} ({organ_of.get(name, '?')})" for name in grid.index],
              fontsize=7)
ax.grid(False)
fig.colorbar(image, ax=ax, label="log10(cells + 1)")
ax.set_title("Cells per cell line and plate")
plt.tight_layout()
plt.show()

**Cell-count response.** All lines share each well, so a drug that kills or stops one line
lowers that line's share of the well's cells. Comparing each line's share with its mean share
in the same plate's DMSO wells gives a whole-dataset view of drug sensitivity from the counts
alone. It is a proxy: proliferation and capture differences show up in it too.

In [ ]:
share = counts.pivot_table(index="sample", columns="cell_line_id", values="n_cells",
                           aggfunc="sum", fill_value=0)
share = share.div(share.sum(axis=1), axis=0)
well_info = wells.loc[share.index]
is_control = well_info["drug"].eq(CONTROL_DRUG)
control_share = share[is_control].groupby(well_info.loc[is_control, "plate"]).mean()
expected = control_share.reindex(well_info["plate"]).set_axis(share.index)
log2_share = np.log2((share + 1e-4) / (expected + 1e-4)).where(expected > 0)

dose = max(DOSES_UM or [5.0])
at_dose = ~is_control & np.isclose(well_info["dose_uM"], dose)
by_drug = log2_share[at_dose].groupby(well_info.loc[at_dose, "drug"]).mean()
organ = lines["organ"].reindex(by_drug.columns)
by_drug = by_drug.loc[:, organ.isin(ORGANS).to_numpy() if ORGANS else slice(None)]
by_drug = by_drug.dropna(axis=1, how="all").dropna(axis=0, how="all")
if by_drug.empty:
    print(f"No treated wells at {dose:g} µM with DMSO wells on the same plate.")
else:
    top = by_drug.mean(axis=1).nsmallest(30).index
    heat = by_drug.loc[top].T
    fig, ax = plt.subplots(figsize=(0.35 * len(top) + 4, 0.3 * len(heat) + 3))
    image = ax.imshow(heat, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
    ax.set_xticks(range(len(top)), top, rotation=90)
    ax.set_yticks(range(len(heat)), lines["cell_name"].reindex(heat.index))
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="log2 share vs DMSO")
    ax.set_title(f"Change in each line's share of the well, {dose:g} µM (most depleting drugs first)")
    plt.tight_layout()
    plt.show()

## 3. Per-cell QC from `obs_metadata`

`obs_metadata` describes every captured cell (about 100.6 million), including the cells that
failed Tahoe's QC filter and are not in the expression data (`pass_filter`). It also gives
genes and transcripts per cell, the mitochondrial fraction and the cell-cycle phase. Reading
every `OBS_ROW_GROUP_STRIDE`-th row group keeps this quick.

In [ ]:
if USE_OBS_METADATA:
    obs_file = pq.ParquetFile(metadata_file("obs_metadata", TAHOE_DIR))
    groups = list(range(0, obs_file.num_row_groups, OBS_ROW_GROUP_STRIDE))
    qc = obs_file.read_row_groups(groups, columns=["plate", "cell_name", "gene_count", "tscp_count",
                                                   "pcnt_mito", "phase", "pass_filter"]).to_pandas()
    print(f"{len(qc):,} of {obs_file.metadata.num_rows:,} cells")
    print(qc["pass_filter"].value_counts().to_string())

In [ ]:
if USE_OBS_METADATA:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, column, log in [(axes[0], "gene_count", True), (axes[1], "tscp_count", True),
                            (axes[2], "pcnt_mito", False)]:
        for status, part in qc.groupby("pass_filter", observed=True):
            values = part[column].to_numpy(dtype=float)
            bins = np.logspace(0, np.log10(values.max() + 1), 60) if log else 60
            ax.hist(values, bins=bins, alpha=0.6, label=str(status))
        ax.set(xscale="log" if log else "linear", title=column, ylabel="cells")
        ax.legend(title="pass_filter")
    plt.tight_layout()
    plt.show()

    by_line = qc.groupby("cell_name", observed=True)
    order = by_line["gene_count"].median().sort_values().index
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, 0.2 * len(order))), sharey=True)
    by_line["gene_count"].median().loc[order].plot.barh(ax=axes[0], title="Median genes per cell")
    (pd.crosstab(qc["cell_name"], qc["phase"], normalize="index").loc[order]
       .plot.barh(stacked=True, ax=axes[1], title="Cell-cycle phase"))
    axes[1].legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

## 4. Select cells

The selection is resolved against the metadata into wells and cell lines. `select_samples`
adds the DMSO wells of the selected wells' plates, so every treatment can be compared with
controls from its own plate. The cells are then picked in two steps:

1. `read_obs` reads the metadata of all matching cells.
2. `sample_per_group` keeps up to `CELLS_PER_GROUP` per cell line and well, and `read_cells`
   reads the counts of only those cells.

In [ ]:
selected_wells = select_samples(sample_meta, drugs=DRUGS, doses_uM=DOSES_UM, plates=PLATES,
                                include_controls=INCLUDE_CONTROLS)
selected_lines = select_cell_lines(cell_line_meta, organs=ORGANS, cell_names=CELL_NAMES)
selection = cells_filter(samples=selected_wells, cell_lines=selected_lines)
matching = counts["sample"].isin(selected_wells) & counts["cell_line_id"].isin(selected_lines)
print(f"{len(selected_wells)} wells x {len(selected_lines)} cell lines: "
      f"{counts.loc[matching, 'n_cells'].sum():,} cells")
wells.loc[selected_wells]

In [ ]:
start = time.time()
obs = read_obs(dataset, selection)
picked = sample_per_group(obs, by=GROUP_BY, n=CELLS_PER_GROUP, seed=SEED)
print(f"{len(obs):,} matching cells, {len(picked):,} picked ({time.time() - start:.0f} s)")
pd.crosstab(picked["cell_line_id"].map(lines["cell_name"]).rename("cell line"), picked["drug"])

In [ ]:
start = time.time()
table = read_cells(dataset, picked["BARCODE_SUB_LIB_ID"], filter=selection)
lookup, var = gene_lookup(gene_meta)
adata = table_to_anndata(table, lookup, var, sample_table(sample_meta), lines)
del table
adata.write_h5ad(OUT_DIR / "tahoe_subset.h5ad")  # raw counts, genes by Ensembl id
print(f"Loaded {adata.n_obs:,} cells x {adata.n_vars:,} genes in {time.time() - start:.0f} s")
adata

## 5. Visualise the subset

Gene symbols replace the Ensembl ids for plotting (the saved `.h5ad` keeps the ids, which TERRA
needs). `treatment` labels each cell with its drug and dose, or `DMSO`.

In [ ]:
adata.var_names = adata.var["gene_symbol"].astype(str).to_numpy()
adata.var_names_make_unique()
dose_label = adata.obs["dose_uM"].map(lambda dose: f"{dose:g} µM")
adata.obs["treatment"] = pd.Categorical(
    np.where(adata.obs["is_control"], "DMSO", adata.obs["drug"].astype(str) + " " + dose_label))
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
for ax, key in zip(axes, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], strict=True):
    sns.boxplot(data=adata.obs, x="cell_name", y=key, ax=ax, showfliers=False)
    ax.set_xlabel("")
axes[-1].tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
adata.layers["counts"] = adata.X.copy()
adata.X = adata.X.astype(np.float32)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_neighbors=15, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.pl.umap(adata, color=["cell_name", "treatment", "plate", "pct_counts_mt"], ncols=2, wspace=0.6)

**Drug effects.** For each cell line, this measures how far a treatment's mean lies from the
mean of the same plate's DMSO cells. It uses the first 20 principal components and divides by
the controls' spread (root-mean-square distance to their mean). A random group of *n* control
cells would score about 1/√*n*.

In [ ]:
pcs = adata.obsm["X_pca"][:, :20]
cells = adata.obs.reset_index(drop=True)
rows = []
for (line, _plate), part in cells.groupby(["cell_name", "plate"], observed=True):
    control = part.index[part["treatment"] == "DMSO"]
    if len(control) < 10:
        continue
    centre = pcs[control].mean(axis=0)
    spread = np.sqrt(((pcs[control] - centre) ** 2).sum(axis=1).mean())
    for treatment, group in part[part["treatment"] != "DMSO"].groupby("treatment", observed=True):
        if len(group) >= 10:
            rows.append({"cell_name": line, "treatment": treatment, "n_cells": len(group),
                         "shift": np.linalg.norm(pcs[group.index].mean(axis=0) - centre) / spread})
effects = pd.DataFrame(rows, columns=["cell_name", "treatment", "n_cells", "shift"])

if effects.empty:
    print("No cell line has at least 10 DMSO and 10 treated cells on one plate.")
else:
    heat = effects.pivot_table(index="cell_name", columns="treatment", values="shift")
    fig, ax = plt.subplots(figsize=(1.2 * heat.shape[1] + 3, 0.35 * heat.shape[0] + 2))
    image = ax.imshow(heat, aspect="auto", cmap="magma")
    ax.set_xticks(range(heat.shape[1]), heat.columns, rotation=45, ha="right")
    ax.set_yticks(range(heat.shape[0]), heat.index)
    ax.grid(False)
    bright = np.nanmin(heat.to_numpy()) + 0.6 * np.ptp(heat.to_numpy()[np.isfinite(heat.to_numpy())])
    for (i, j), value in np.ndenumerate(heat.to_numpy()):
        if np.isfinite(value):
            ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=8,
                    color="black" if value > bright else "white")
    fig.colorbar(image, ax=ax, label="shift from DMSO (control spread)")
    ax.set_title("Treatment effect per cell line")
    plt.tight_layout()
    plt.show()

In [ ]:
if not effects.empty:
    line, treatment = effects.loc[effects["shift"].idxmax(), ["cell_name", "treatment"]]
    pair = adata[(adata.obs["cell_name"] == line)
                 & adata.obs["treatment"].isin([treatment, "DMSO"])].copy()
    pair.obs["treatment"] = pair.obs["treatment"].cat.remove_unused_categories()
    sc.tl.rank_genes_groups(pair, "treatment", groups=[treatment], reference="DMSO",
                            method="wilcoxon")
    top_genes = sc.get.rank_genes_groups_df(pair, group=treatment).head(10)
    print(f"Strongest response: {treatment} in {line}")
    sc.pl.rank_genes_groups(pair, n_genes=20, sharey=False)
    sc.pl.dotplot(adata[adata.obs["cell_name"] == line], top_genes["names"].tolist(),
                  groupby="treatment")
    print(top_genes.to_string(index=False))

## 6. Export

`tahoe_subset.h5ad` (written in section 4) holds the raw counts with the metadata. With
`TERRA_MODEL_DIR` set, the same cells are also tokenized as cell-only records, with the label
columns that multi-stream fine-tuning routes on (`lineage_value`, `is_control_value`, ...). The
layout is `TERRA_LAYOUT`. This is the local equivalent of `python -m terra.tahoe.prepare tahoe`.
Label the spatial dataset with the `label_codes.json` written here (`prepare spatial
--label-codes`), then fine-tune with `configs/finetune/tahoe_alignment.yaml`.

In [ ]:
if TERRA_MODEL_DIR is None:
    print("TERRA_MODEL_DIR is not set: skipping the tokenized TERRA dataset.")
else:
    from terra.tahoe.prepare import write_tahoe_dataset

    start = time.time()
    raw = sc.read_h5ad(OUT_DIR / "tahoe_subset.h5ad")
    tahoe_ds = write_tahoe_dataset(raw, TERRA_MODEL_DIR, LINEAGE_MAP, OUT_DIR / "terra",
                                   data_overrides=TERRA_LAYOUT)
    print(f"Tokenized {raw.n_obs:,} cells into {tahoe_ds} in {time.time() - start:.0f} s")